# Sync-free eye ellipse pipeline (DLC → ellipses → verify → Kerr → map)

This notebook builds **per-video**, **frame-indexed** eye artifacts next to each eye video
(`eye_videos/LE` or `RE`), **without** coupling to `final_sync_df`. Open Ephys time is applied
only in the final optional mapping step.

## Frame index contract

- `eye_frame` matches OpenCV playback: `current_idx = CAP_PROP_POS_FRAMES - 1` (same as
  `interactive_ellipse_corrector` in `data_verification_utils.py`).
- `BlockSync.eye_tracking_analysis` fits DLC rows with indices `1 .. len(data)-2` only; frames
  `0` and `len(data)-1` are **NaN placeholders** for ellipse columns (same convention as the
  merge in `read_dlc_data`).

## Outputs (next to the eye `.mp4`)

- `{left|right}_syncfree_{tag}_ellipses.csv` — one row per video frame
- `{left|right}_syncfree_{tag}_meta.json` — DLC path, threshold, frame counts
- After verification: `{left|right}_syncfree_{tag}_verified.csv` and `{left|right}_syncfree_{tag}_kerr_refs.csv`
- After Kerr: `{left|right}_syncfree_{tag}_kerr_angles.csv`, `{left|right}_syncfree_{tag}_degrees.csv`
- After mapping: `analysis/{left|right}_eye_degrees_from_syncfree_{tag}.csv`

## Staleness guard

If you regenerate `final_sync_df.csv`, rerun **only** the mapping section (or rerun Kerr+map
if ellipse data changed).

## Prerequisites

- Block folder layout per `BlockSync` / main README
- DeepLabCut CSV in each eye folder (same naming rules as `read_dlc_data`)


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from eye_tracking_system_tools.preprocessing import utility_functions as uf
from eye_tracking_system_tools.preprocessing.BlockSync_class import BlockSync
from eye_tracking_system_tools.preprocessing.data_verification_utils import interactive_ellipse_corrector
from eye_tracking_system_tools.preprocessing.sync_free_eye_io import (
    append_kerr_columns_syncfree,
    default_syncfree_paths,
    map_syncfree_degrees_to_final_sync,
    run_syncfree_ellipses_for_eye,
    write_meta_json,
    write_self_kerr_refs_csv,
)


In [ ]:
# --- Parameters (edit) ---
experiment_path = Path(r"D:\sample_data_for_eye_repo")
animal_call = "PV_126"
block_numbers = [6]
bad_blocks = []

uncertainty_thr = 0.95
artifact_tag = "v1"

RUN_ELLIPSES = True
RUN_VERIFY = False
RUN_KERR = True
RUN_MAP = True


In [ ]:
blocks = uf.block_generator(
    block_numbers=block_numbers,
    experiment_path=experiment_path,
    animal=animal_call,
    bad_blocks=bad_blocks,
)
if not blocks:
    raise ValueError("No blocks matched the parameters.")
print("Blocks:", [str(b.block_path) for b in blocks])


In [ ]:
def _video_for(block, eye: str) -> Path:
    if not getattr(block, "le_videos", None) or not getattr(block, "re_videos", None):
        block.handle_eye_videos()
    return Path(block.le_videos[0] if eye.lower() == "left" else block.re_videos[0])


def _merge_self_kerr_refs(block, eye: str, rx: int, ry: int) -> None:
    ap = Path(block.analysis_path)
    p = ap / "self_kerr_refs.csv"
    row = {
        "kerr_ref_l_x": getattr(block, "kerr_ref_l_x", None),
        "kerr_ref_l_y": getattr(block, "kerr_ref_l_y", None),
        "kerr_ref_r_x": getattr(block, "kerr_ref_r_x", None),
        "kerr_ref_r_y": getattr(block, "kerr_ref_r_y", None),
    }
    if p.exists():
        prev = pd.read_csv(p).iloc[0].to_dict()
        for k in row:
            if k in prev and pd.notna(prev[k]):
                row[k] = prev[k]
    if eye == "left":
        row["kerr_ref_l_x"], row["kerr_ref_l_y"] = rx, ry
        block.kerr_ref_l_x, block.kerr_ref_l_y = rx, ry
    else:
        row["kerr_ref_r_x"], row["kerr_ref_r_y"] = rx, ry
        block.kerr_ref_r_x, block.kerr_ref_r_y = rx, ry
    write_self_kerr_refs_csv(
        block,
        kerr_ref_l_x=row["kerr_ref_l_x"],
        kerr_ref_l_y=row["kerr_ref_l_y"],
        kerr_ref_r_x=row["kerr_ref_r_x"],
        kerr_ref_r_y=row["kerr_ref_r_y"],
    )


def _maybe_load_kerr_refs(block, eye: str):
    ap = Path(block.analysis_path)
    p = ap / "self_kerr_refs.csv"
    if eye == "left":
        if getattr(block, "kerr_ref_l_x", None) is not None and getattr(block, "kerr_ref_l_y", None) is not None:
            return int(block.kerr_ref_l_x), int(block.kerr_ref_l_y)
    else:
        if getattr(block, "kerr_ref_r_x", None) is not None and getattr(block, "kerr_ref_r_y", None) is not None:
            return int(block.kerr_ref_r_x), int(block.kerr_ref_r_y)
    if p.exists():
        row = pd.read_csv(p).iloc[0]
        if eye == "left" and pd.notna(row.get("kerr_ref_l_x")):
            return int(row["kerr_ref_l_x"]), int(row["kerr_ref_l_y"])
        if eye == "right" and pd.notna(row.get("kerr_ref_r_x")):
            return int(row["kerr_ref_r_x"]), int(row["kerr_ref_r_y"])
    raise FileNotFoundError(
        f"No Kerr ref for {eye}. Run verification (RUN_VERIFY=True) or create {p}"
    )


for block in blocks:
    try:
        block.sample_rate = float(block.get_sample_rate())
    except Exception:
        pass

    for eye in ("left", "right"):
        vid = _video_for(block, eye)
        paths = default_syncfree_paths(vid, eye, artifact_tag)

        if RUN_ELLIPSES:
            df_ell, meta = run_syncfree_ellipses_for_eye(block, eye, uncertainty_thr)
            meta["artifact_paths"] = {k: str(v) for k, v in paths.items()}
            df_ell.to_csv(paths["ellipses"], index=False)
            write_meta_json(meta, paths["meta"])
            print(f"[{eye}] wrote {paths['ellipses']}")

        if RUN_VERIFY:
            df_in = pd.read_csv(paths["ellipses"])

            def on_save(df_corr, ref):
                df_corr.to_csv(paths["verified"], index=False)
                if ref is not None:
                    rx, ry = int(ref[0]), int(ref[1])
                    pd.DataFrame(
                        [{"eye": eye, "kerr_ref_x": rx, "kerr_ref_y": ry}]
                    ).to_csv(paths["kerr_refs_sidecar"], index=False)
                    _merge_self_kerr_refs(block, eye, rx, ry)
                print(f"[{eye}] saved verified (+ refs if picked)")

            interactive_ellipse_corrector(df_in, vid, eye, on_save=on_save)

        if RUN_KERR:
            src = paths["verified"] if paths["verified"].exists() else paths["ellipses"]
            df_kin = pd.read_csv(src)
            kx, ky = _maybe_load_kerr_refs(block, eye)
            merged_deg, angles, fz = append_kerr_columns_syncfree(df_kin, kx, ky)
            angles.to_csv(paths["kerr_raw"], index=False)
            merged_deg.to_csv(paths["degrees"], index=False)
            print(f"[{eye}] Kerr f_z={fz:.6g}; wrote {paths['degrees']}")

        if RUN_MAP:
            deg_path = paths["degrees"]
            if not deg_path.exists():
                raise FileNotFoundError(f"Missing degrees file: {deg_path}")
            df_deg = pd.read_csv(deg_path)
            mapped = map_syncfree_degrees_to_final_sync(block, df_deg, eye)
            out_name = f"{eye}_eye_degrees_from_syncfree_{artifact_tag}.csv"
            outp = Path(block.analysis_path) / out_name
            Path(block.analysis_path).mkdir(parents=True, exist_ok=True)
            mapped.to_csv(outp, index=False)
            print(f"[{eye}] mapped -> {outp}")


In [ ]:
# --- Validation (optional) ---
# b = blocks[0]
# from eye_tracking_system_tools.preprocessing.block_sync_core import load_final_sync_df
# fs = load_final_sync_df(b, verbose=False)
# L = set(pd.to_numeric(fs["L_eye_frame"], errors="coerce").dropna().astype(int))
# deg = pd.read_csv(Path(b.analysis_path) / f"left_eye_degrees_from_syncfree_{artifact_tag}.csv")
# E = set(pd.to_numeric(deg["eye_frame"], errors="coerce").dropna().astype(int))
# print("missing frames (final_sync not in mapped):", len(L - E), sorted(list(L - E))[:20])
